# Lab 8 · Notebook 3 (advanced): Statistics the hacker's way

**Optional. Time:** about 40 minutes. **What you will do:** compute a p-value and a confidence interval **without any formula**, using only a loop and random numbers. This is the approach from Jake VanderPlas's talk *Statistics for Hackers*, linked in Lecture 8:

> If you can write a for loop, you can do statistics.

Two ideas:

- **Permutation test:** if the label (skin or not) does not matter, shuffling the labels should not change the difference. So shuffle them many times and see how often chance gives a difference as big as the real one. That fraction **is** the p-value.
- **Bootstrap:** resample your own data with replacement, many times, to see how much your result would wobble if you repeated the experiment. That gives a confidence interval.

You need notebook 2 first. Every **Write it** cell has a self-check; because the answers use random numbers, the checks accept a range.

The [talk is on YouTube](https://www.youtube.com/watch?v=Iq9DzN6mvYA) (PyCon 2016, 40 minutes) if you want the full story.

In [ ]:
# Run this first. It loads the DepMap table and makes a random number generator.
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import mannwhitneyu, spearmanr

FILE = "depmap.csv"
URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab08/data/" + FILE

if not Path(FILE).exists():
    urllib.request.urlretrieve(URL, FILE)

df = pd.read_csv(FILE)
skin = df[df["oncotree_lineage"] == "Skin"]
rest = df[df["oncotree_lineage"] != "Skin"]

# one True/False per cell line: is it skin?
is_skin = (df["oncotree_lineage"] == "Skin").to_numpy()

# the random number generator: rng.permutation shuffles, rng.choice picks at random
rng = np.random.default_rng()
print("cell lines:", len(is_skin), "  skin:", is_skin.sum())

## 1. One number for the difference

A permutation test needs a function that turns **values plus labels** into one number. We use the median difference, as in the lecture.

With numpy arrays, `values[labels]` keeps the values where `labels` is True, and `values[~labels]` the others (`~` flips True and False).

### Write it

**Your task:** write `median_diff(values, labels)`. It returns the median of the values where `labels` is True minus the median of the values where it is False. Use `np.median`.

In [ ]:
# TODO - the median of the True group minus the median of the False group
def median_diff(values, labels):



# --- self-check (do not edit) ---
sox10 = df["SOX10"].to_numpy()
real = median_diff(sox10, is_skin)
assert real is not None, "median_diff returned nothing. Add a return line"
assert abs(real - (skin["SOX10"].median() - rest["SOX10"].median())) < 1e-9, f"median_diff gives {real}. It should be the median of values[labels] minus the median of values[~labels]"
assert abs(median_diff(np.array([1.0, 2.0, 10.0, 20.0]), np.array([False, False, True, True])) - 13.5) < 1e-9, "Check the order: True group minus False group"
print("Correct: the real SOX10 difference is", round(real, 2))

## 2. Shuffle the labels

`rng.permutation(is_skin)` returns the same 73 Trues and 1,092 Falses in a random order. The skin label now lands on 73 random cell lines. Any difference you measure is pure chance.

In [ ]:
# DEMO - uses your median_diff: one shuffle, then a thousand
sox10 = df["SOX10"].to_numpy()
real = median_diff(sox10, is_skin)

shuffled = rng.permutation(is_skin)
print("real difference:    ", round(real, 3))
print("one shuffled label: ", round(median_diff(sox10, shuffled), 3))

null = []
for i in range(1000):
    null.append(median_diff(sox10, rng.permutation(is_skin)))

fig, ax = plt.subplots()
sns.histplot(x=null, bins=30, ax=ax)
ax.axvline(real, color="red", label="real skin difference")
ax.set_xlabel("median difference with shuffled labels")
ax.legend()
plt.show()

The histogram is the **null distribution**: what the difference looks like when the label means nothing. The p-value is the fraction of shuffles that came out at least as extreme as the real difference. We use `abs()` so a big difference in either direction counts.

For SOX10 that fraction is 0 out of 1,000. We report it as **p < 0.001**: with 1,000 shuffles you cannot see anything rarer than 1 in 1,000.

### Write it

**Your task:** write `permutation_p(values, labels, n=1000)`. It shuffles the labels `n` times, counts how often `abs(median_diff(values, shuffled))` is at least `abs` of the real difference, and returns `count / n`.

In [ ]:
# TODO - a permutation test, from a loop
def permutation_p(values, labels, n=1000):



# --- self-check (do not edit) ---
p_sox10 = permutation_p(df["SOX10"].to_numpy(), is_skin)
assert p_sox10 is not None, "permutation_p returned nothing. End with return count / n"
assert p_sox10 < 0.01, f"SOX10 gave p = {p_sox10}. Count the shuffles where abs(shuffled difference) >= abs(real difference)"
p_ran = permutation_p(df["RAN"].to_numpy(), is_skin)
assert 0.4 < p_ran < 0.85, f"RAN gave p = {p_ran}, expected about 0.6. Shuffle with rng.permutation(labels) inside the loop, and divide the count by n"
print("Correct: SOX10 p =", p_sox10, " RAN p =", p_ran)

### Write it

**Your task:** compare your permutation test with Mann–Whitney from notebook 2 for `MDM2` and `RAN`. Store the four p-values in `perm_mdm2`, `mw_mdm2`, `perm_ran` and `mw_ran`, and print them.

In [ ]:
# TODO - permutation vs Mann-Whitney, MDM2 and RAN



# --- self-check (do not edit) ---
assert perm_mdm2 < 0.01, "MDM2 should give a very small permutation p. Use permutation_p(df[\"MDM2\"].to_numpy(), is_skin)"
assert abs(mw_mdm2 - mannwhitneyu(skin["MDM2"], rest["MDM2"]).pvalue) < 1e-15, "mw_mdm2 should be the Mann-Whitney p-value for MDM2"
assert 0.4 < perm_ran < 0.85, f"perm_ran is {perm_ran}, expected about 0.6"
assert abs(mw_ran - mannwhitneyu(skin["RAN"], rest["RAN"]).pvalue) < 1e-12, "mw_ran should be the Mann-Whitney p-value for RAN"
print("Correct: a loop and a formula, the same verdict both times")

## 3. The bootstrap: how sure is the number?

A p-value answers *could it be chance?* It does not tell you how precisely you know the difference. For that you want a **confidence interval**: a range that probably contains the true difference.

The bootstrap gets it from your own data. Pretend your 73 skin lines are the whole world, and draw 73 from them **with replacement**: some lines get picked twice, some not at all. Do the same for the rest. Compute the difference. Repeat 1,000 times. The middle 95 % of those differences is the 95 % confidence interval.

```python
rng.choice(values, size=len(values), replace=True)    # one bootstrap sample
np.percentile(boots, [2.5, 97.5])                     # the middle 95 %
```

In [ ]:
# DEMO - a bootstrap interval for the SOX10 difference
a = skin["SOX10"].to_numpy()
b = rest["SOX10"].to_numpy()

boots = []
for i in range(1000):
    a_star = rng.choice(a, size=len(a), replace=True)
    b_star = rng.choice(b, size=len(b), replace=True)
    boots.append(np.median(a_star) - np.median(b_star))

low, high = np.percentile(boots, [2.5, 97.5])
print(f"SOX10 difference {np.median(a) - np.median(b):.2f}, 95% interval [{low:.2f}, {high:.2f}]")

fig, ax = plt.subplots()
sns.histplot(x=boots, bins=30, ax=ax)
ax.axvline(low, color="grey", linestyle="--")
ax.axvline(high, color="grey", linestyle="--")
ax.set_xlabel("bootstrapped median difference")
plt.show()

"SOX10 difference -1.22, 95 % interval [-1.43, -1.00]" says more than "p < 0.001": the difference is real **and** it is big, somewhere between -1.0 and -1.4. If an interval includes 0, the data cannot rule out no difference at all.

### Broken

This bootstrap gives an interval with no width at all. Run it, then find the one word that is wrong.

In [ ]:
# BROKEN - the interval has no width
a = skin["MDM2"].to_numpy()
b = rest["MDM2"].to_numpy()
boots = []
for i in range(1000):
    a_star = rng.choice(a, size=len(a), replace=False)
    b_star = rng.choice(b, size=len(b), replace=False)
    boots.append(np.median(a_star) - np.median(b_star))
low, high = np.percentile(boots, [2.5, 97.5])
print(f"MDM2 95% interval [{low:.2f}, {high:.2f}]")


# --- self-check (do not edit) ---
assert high - low > 0.05, "The interval has no width. Without replacement, drawing all 73 lines just reorders them, so the median never changes. Use replace=True"
assert high < 0, "The whole MDM2 interval should be below 0"
print("Fixed: with replacement, every sample is a little different")

### Write it

**Your task:** wrap the bootstrap in a function `bootstrap_ci(a, b, n=1000)`. `a` and `b` are two Series. It returns `(low, high)`, the 95 % interval of the median difference `a` minus `b`. Then use it on SOX10 and on TP53.

In [ ]:
# TODO - a bootstrap confidence interval, as a function
def bootstrap_ci(a, b, n=1000):



# --- self-check (do not edit) ---
result = bootstrap_ci(skin["SOX10"], rest["SOX10"])
assert result is not None and len(result) == 2, "bootstrap_ci should return two numbers: return low, high"
low, high = result
assert -1.65 < low < -1.25 and -1.15 < high < -0.85, f"SOX10 gave [{low:.2f}, {high:.2f}], expected about [-1.43, -1.00]. Resample with replace=True, then np.percentile(boots, [2.5, 97.5])"
t_low, t_high = bootstrap_ci(skin["TP53"], rest["TP53"])
assert t_low > 0, f"The TP53 interval [{t_low:.2f}, {t_high:.2f}] should lie above 0"
print(f"Correct: SOX10 [{low:.2f}, {high:.2f}], TP53 [{t_low:.2f}, {t_high:.2f}]")

## 4. The same trick works for a correlation

In notebook 2, Spearman r for MITF vs SOX10 in skin was 0.46. Is that more than chance? Shuffle **one** column: that breaks any link between the two, while keeping both columns' values. Then count how often the shuffled r is at least as big as the real one.

### Write it

**Your task:** shuffle `skin["MITF"]` 1,000 times with `rng.permutation`, compute `spearmanr(shuffled, skin["SOX10"]).statistic` each time, and store in `p_corr` the fraction of shuffles where `abs` of the shuffled r is at least `abs` of the real r.

In [ ]:
# TODO - a permutation test for a correlation



# --- self-check (do not edit) ---
assert 0 <= p_corr < 0.01, f"p_corr is {p_corr}. Count shuffles where abs(shuffled r) >= abs(real r), then divide by 1000"
print("Correct: chance almost never gives r = 0.46 in 73 lines. scipy's own p-value:", f"{spearmanr(skin['MITF'], skin['SOX10']).pvalue:.1g}")

## 5. When the methods disagree

For big differences (SOX10, MDM2) and for no difference (RAN), every method gives the same verdict. For **tiny** differences they can disagree, because they ask slightly different questions. Run this:

In [ ]:
# DEMO - uses your three functions: NRAS, a tiny difference, three answers
nras = df["NRAS"].to_numpy()
print("median difference:     ", round(median_diff(nras, is_skin), 3))
print("Mann-Whitney p:        ", round(mannwhitneyu(skin["NRAS"], rest["NRAS"]).pvalue, 3))
print("permutation p, medians:", permutation_p(nras, is_skin))
low, high = bootstrap_ci(skin["NRAS"], rest["NRAS"])
print(f"bootstrap 95% interval: [{low:.3f}, {high:.3f}]")

Mann–Whitney compares the **ranks** of all the values and says p ≈ 0.04. The permutation test and the bootstrap look only at the **medians**, and say the difference could be chance (p ≈ 0.12 to 0.15, and an interval whose top end sits right at 0).

Neither is wrong. The lesson from notebook 2 holds: a difference of -0.03 is too small to matter, whatever p says. When a verdict flips depending on the method, the result is fragile. Say so, and look at the size of the difference.

## If you finish early

1. Change `median_diff` to use `np.mean` and rerun the NRAS cell. Which methods agree now?
2. Run `permutation_p` with `n=10000`. How much does the RAN p-value move between runs with 1,000 and with 10,000 shuffles?
3. Write `bootstrap_r(x, y, n=1000)`: resample **rows** (pick random positions with `rng.integers(0, len(x), len(x))` and use them for both columns), and return a 95 % interval for Spearman r of MITF vs SOX10.